In [9]:
# ============================================================
# CELL 1 — CONFIGURATION
# ============================================================

import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F

# Fabric Warehouse
WAREHOUSE = "Siddarth_Weather_Warehouse"

TARGET_TABLE = f"{WAREHOUSE}.dbo.WeatherForecast"
STAGING_TABLE = f"{WAREHOUSE}.dbo.WeatherForecast_Stage"

# Location
LOCATION = "Hyderabad"
LATITUDE = 17.3850
LONGITUDE = 78.4867
TIMEZONE = "Asia/Kolkata"

# Open-Meteo
FORECAST_DAYS = 7
OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"

print("Configuration loaded.")
print(f"Warehouse : {WAREHOUSE}")
print(f"Location  : {LOCATION}")
print(f"Latitude  : {LATITUDE}")
print(f"Longitude : {LONGITUDE}")

StatementMeta(, c25b3c1d-6a1a-4deb-ba1b-ba4fc96f032f, 11, Finished, Available, Finished, False)

Configuration loaded.
Warehouse : Siddarth_Weather_Warehouse
Location  : Hyderabad
Latitude  : 17.385
Longitude : 78.4867


In [10]:
# ============================================================
# CELL 2 — CALL OPEN-METEO API (with retry)
# ============================================================

import time

MAX_RETRIES = 3

params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "hourly": ",".join([
        "temperature_2m",
        "relative_humidity_2m",
        "apparent_temperature",
        "precipitation",
        "rain",
        "weather_code",
        "cloud_cover",
        "pressure_msl",
        "wind_speed_10m",
        "wind_direction_10m",
        "wind_gusts_10m"
    ]),
    "forecast_days": FORECAST_DAYS,
    "timezone": TIMEZONE
}

def fetch_weather(params):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(OPEN_METEO_URL, params=params, timeout=30)
            response.raise_for_status()
            payload = response.json()
            if "hourly" not in payload or "time" not in payload["hourly"]:
                raise ValueError("API response is missing the 'hourly' block.")
            return payload
        except Exception as error:
            last_error = error
            print(f"Attempt {attempt}/{MAX_RETRIES} failed: {error}")
            if attempt < MAX_RETRIES:
                time.sleep(5 * attempt)
    raise RuntimeError(f"Open-Meteo request failed after {MAX_RETRIES} attempts: {last_error}")

weather_json = fetch_weather(params)

print("Open-Meteo API request successful.")
print(f"Latitude : {weather_json['latitude']}")
print(f"Longitude: {weather_json['longitude']}")
print(f"Timezone : {weather_json['timezone']}")
print(f"Elevation: {weather_json['elevation']} m")

StatementMeta(, c25b3c1d-6a1a-4deb-ba1b-ba4fc96f032f, 12, Finished, Available, Finished, False)

Open-Meteo API request successful.
Latitude : 17.398945
Longitude: 78.457085
Timezone : Asia/Kolkata
Elevation: 505.0 m


In [11]:
# ============================================================
# CELL 3 — CREATE SPARK DATAFRAME
# ============================================================

hourly = weather_json["hourly"]

rows = []

for i in range(len(hourly["time"])):
    rows.append({
        "location": LOCATION,
        "latitude": float(LATITUDE),
        "longitude": float(LONGITUDE),
        "timezone": TIMEZONE,

        "weather_time": hourly["time"][i],

        "temperature_c": hourly["temperature_2m"][i],
        "humidity_percent": hourly["relative_humidity_2m"][i],
        "apparent_temperature_c": hourly["apparent_temperature"][i],

        "precipitation_mm": hourly["precipitation"][i],
        "rain_mm": hourly["rain"][i],

        "weather_code": hourly["weather_code"][i],

        "cloud_cover_percent": hourly["cloud_cover"][i],
        "pressure_msl_hpa": hourly["pressure_msl"][i],

        "wind_speed_kmh": hourly["wind_speed_10m"][i],
        "wind_direction_degrees": hourly["wind_direction_10m"][i],
        "wind_gusts_kmh": hourly["wind_gusts_10m"][i],

        "ingested_at": datetime.now(timezone.utc)
    })

weather_df = spark.createDataFrame(rows)

# Convert Open-Meteo's timestamp string into Spark timestamp
weather_df = weather_df.withColumn(
    "weather_time",
    F.to_timestamp("weather_time")
)

print(f"Rows extracted: {weather_df.count()}")

display(weather_df)

StatementMeta(, c25b3c1d-6a1a-4deb-ba1b-ba4fc96f032f, 13, Finished, Available, Finished, False)

Rows extracted: 168


SynapseWidget(Synapse.DataFrame, fa1799de-60b2-4b0d-9d5d-f885b548a6d7)

In [12]:
# ============================================================
# CELL 4 — DATA QUALITY VALIDATION
# ============================================================

# 1. Check required columns
required_columns = [
    "location",
    "weather_time",
    "temperature_c",
    "humidity_percent"
]

missing_columns = [
    column for column in required_columns
    if column not in weather_df.columns
]

if missing_columns:
    raise Exception(
        f"Missing required columns: {missing_columns}"
    )

print("✓ Required columns present")


# 2. Check duplicate business keys
# Business key = location + weather_time

duplicate_count = (
    weather_df
    .groupBy("location", "weather_time")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_count > 0:
    raise Exception(
        f"Found {duplicate_count} duplicate weather records."
    )

print("✓ No duplicate location + weather_time records")


# 3. Check null business keys

null_key_count = (
    weather_df
    .filter(
        F.col("location").isNull() |
        F.col("weather_time").isNull()
    )
    .count()
)

if null_key_count > 0:
    raise Exception(
        f"Found {null_key_count} records with null business keys."
    )

print("✓ No null business keys")


# 4. Display schema

print("\nSpark DataFrame schema:")
weather_df.printSchema()

# Row count must match the forecast window
expected_rows = FORECAST_DAYS * 24
actual_rows = weather_df.count()
if actual_rows != expected_rows:
    raise Exception(f"Expected {expected_rows} rows, got {actual_rows}.")
print("✓ Row count matches forecast window")

# Value range checks
range_violations = weather_df.filter(
    F.col("temperature_c").isNull()
    | F.col("humidity_percent").isNull()
    | (F.col("temperature_c") < -90) | (F.col("temperature_c") > 60)
    | (F.col("humidity_percent") < 0) | (F.col("humidity_percent") > 100)
    | (F.col("cloud_cover_percent") < 0) | (F.col("cloud_cover_percent") > 100)
    | (F.col("precipitation_mm") < 0)
).count()
if range_violations > 0:
    raise Exception(f"{range_violations} rows failed value range checks.")
print("✓ Value ranges valid")


# 5. Summary

print("\nValidation summary:")
print(f"Total records : {weather_df.count():,}")
print(f"Location      : {LOCATION}")
print(f"Forecast days : {FORECAST_DAYS}")

StatementMeta(, c25b3c1d-6a1a-4deb-ba1b-ba4fc96f032f, 14, Finished, Available, Finished, False)

✓ Required columns present
✓ No duplicate location + weather_time records
✓ No null business keys

Spark DataFrame schema:
root
 |-- apparent_temperature_c: double (nullable = true)
 |-- cloud_cover_percent: long (nullable = true)
 |-- humidity_percent: long (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- location: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- pressure_msl_hpa: double (nullable = true)
 |-- rain_mm: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- timezone: string (nullable = true)
 |-- weather_code: long (nullable = true)
 |-- weather_time: timestamp (nullable = true)
 |-- wind_direction_degrees: long (nullable = true)
 |-- wind_gusts_kmh: double (nullable = true)
 |-- wind_speed_kmh: double (nullable = true)

✓ Row count matches forecast window
✓ Value ranges valid

Validation summary:
Total records : 168
Loca

In [13]:
# ============================================================
# FABRIC WAREHOUSE SPARK CONNECTOR
# ============================================================

import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

print("Fabric Spark Warehouse connector imported.")

StatementMeta(, c25b3c1d-6a1a-4deb-ba1b-ba4fc96f032f, 15, Finished, Available, Finished, False)

Fabric Spark Warehouse connector imported.


In [14]:
# ============================================================
# LOAD TO WAREHOUSE (staging -> compare -> upsert -> verify)
# ============================================================
INITIAL_LOAD = False   # set True ONLY once, if the target table does not exist yet

keys = ["location", "weather_time"]

# 1. Stage the latest API pull
weather_df.write.mode("overwrite").synapsesql(STAGING_TABLE)
stage_df = spark.read.synapsesql(STAGING_TABLE).cache()
stage_count = stage_df.count()

if INITIAL_LOAD:
    stage_df.write.mode("overwrite").synapsesql(TARGET_TABLE)
    print(f"✓ Initial load complete: {stage_count:,} rows")
else:
    # Fails loudly if the target table is missing (it will NOT silently recreate it)
    target_df = spark.read.synapsesql(TARGET_TABLE).cache()
    previous_count = target_df.count()

    # Existing rows with keys not in the new pull are kept
    kept_df = target_df.join(stage_df.select(*keys), on=keys, how="left_anti")
    new_count = stage_df.join(target_df.select(*keys), on=keys, how="left_anti").count()

    # Everything in the new pull replaces or adds
    final_df = kept_df.unionByName(stage_df).cache()
    final_count = final_df.count()   # materialize BEFORE overwriting the target

    # Safety checks before touching the target table
    duplicate_keys = final_df.groupBy(*keys).count().filter("count > 1").count()
    if duplicate_keys > 0:
        raise Exception(f"UPSERT ABORTED: {duplicate_keys} duplicate keys.")
    if final_count < previous_count:
        raise Exception(f"UPSERT ABORTED: would drop rows ({previous_count} -> {final_count}).")

    final_df.write.mode("overwrite").synapsesql(TARGET_TABLE)

    # Verify what actually landed
    verify_count = spark.read.synapsesql(TARGET_TABLE).count()
    if verify_count != final_count:
        raise Exception(f"VERIFY FAILED: wrote {final_count}, read back {verify_count}.")

    print(f"✓ Upsert complete | previous: {previous_count:,} | new: {new_count:,} | final: {final_count:,}")
    final_df.unpersist()
    target_df.unpersist()

stage_df.unpersist()

spark.read.synapsesql(TARGET_TABLE).select(
    F.min("weather_time").alias("earliest"),
    F.max("weather_time").alias("latest"),
    F.count("*").alias("total_rows")
).show()

StatementMeta(, c25b3c1d-6a1a-4deb-ba1b-ba4fc96f032f, 16, Finished, Available, Finished, False)

✓ Upsert complete | previous: 168 | new: 0 | final: 168
+-------------------+-------------------+----------+
|           earliest|             latest|total_rows|
+-------------------+-------------------+----------+
|2026-10-24 00:00:00|2026-10-30 23:00:00|       168|
+-------------------+-------------------+----------+



In [15]:
# ============================================================
# CELL 6 — VERIFY WAREHOUSE DATA
# ============================================================

warehouse_df = spark.read.synapsesql(TARGET_TABLE)

print(f"Warehouse rows: {warehouse_df.count():,}")

warehouse_df.orderBy("weather_time").show(10, truncate=False)

StatementMeta(, c25b3c1d-6a1a-4deb-ba1b-ba4fc96f032f, 17, Finished, Available, Finished, False)

Warehouse rows: 168
+----------------+--------------------------+----------------+-------------------+---------+--------------+--------+------------+----------------------+---------+----------------+------------+-------------+-------+--------------+-------------------+----------------------+
|humidity_percent|ingested_at               |pressure_msl_hpa|cloud_cover_percent|location |wind_speed_kmh|latitude|timezone    |wind_direction_degrees|longitude|precipitation_mm|weather_code|temperature_c|rain_mm|wind_gusts_kmh|weather_time       |apparent_temperature_c|
+----------------+--------------------------+----------------+-------------------+---------+--------------+--------+------------+----------------------+---------+----------------+------------+-------------+-------+--------------+-------------------+----------------------+
|56              |2026-10-24 11:08:13.285622|1012.3          |59                 |Hyderabad|5.4           |17.385  |Asia/Kolkata|56                    |78.4867  

In [16]:
# ============================================================
# CELL 7 — CHECK WAREHOUSE DATA
# ============================================================

warehouse_df.select(
    F.min("weather_time").alias("earliest_forecast"),
    F.max("weather_time").alias("latest_forecast"),
    F.count("*").alias("total_records")
).show()

StatementMeta(, c25b3c1d-6a1a-4deb-ba1b-ba4fc96f032f, 18, Finished, Available, Finished, False)

+-------------------+-------------------+-------------+
|  earliest_forecast|    latest_forecast|total_records|
+-------------------+-------------------+-------------+
|2026-10-24 00:00:00|2026-10-30 23:00:00|          168|
+-------------------+-------------------+-------------+



In [17]:
warehouse_df = spark.read.synapsesql(
    "Siddarth_Weather_Warehouse.dbo.WeatherForecast"
)

display(
    warehouse_df
    .orderBy("weather_time")
    .limit(10)
)

StatementMeta(, c25b3c1d-6a1a-4deb-ba1b-ba4fc96f032f, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c36d0c70-bcef-40d7-ac8b-d54dfeb2949c)